# Hanoi Land Law — embedding offline trên Colab CPU

Chọn **Runtime → Change runtime type → CPU**. Không cần GPU, Gemini key hoặc Supabase key.

Notebook nhận `colab-e5-input.zip` do repository xuất ra, dùng E5-small ONNX Q8 đã khóa checksum, rồi lưu checkpoint và kết quả vào Google Drive. Colab Free có thể ngắt phiên; mở lại notebook, upload cùng bundle và chạy lại để tiếp tục. Đừng đổi model hoặc sửa bundle giữa các lần chạy.

Đây là pipeline tạo vector; hoàn thành notebook chưa đồng nghĩa chatbot đã được phát hành hoặc đạt nghiệm thu. Không upload `.env`, hồ sơ cá nhân hoặc khóa API.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'onnxruntime>=1.20,<2', 'sentencepiece>=0.2,<0.3', 'numpy>=2,<3'], check=True)


## 1. Kết nối Google Drive để giữ checkpoint qua lần ngắt phiên


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Upload bundle đầu vào

Chọn duy nhất `colab-e5-input.zip`. Bundle chỉ chứa các đoạn tài liệu công khai, manifest và worker đã kiểm chứng.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, zipfile
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Chọn duy nhất colab-e5-input.zip')
BUNDLE = Path('/content/colab-e5-input.zip')
BUNDLE.write_bytes(next(iter(uploaded.values())))
del uploaded
EXPECTED_WORKER_SHA256 = '29ecb6a9baef4c164d2c17dde9af9e76a476f6f8e5f8a768000d3bd12ee850db'
with zipfile.ZipFile(BUNDLE) as archive:
    if archive.getinfo('offline_e5_colab.py').file_size > 100000:
        raise ValueError('Worker quá lớn')
    worker_bytes = archive.read('offline_e5_colab.py')
    if hashlib.sha256(worker_bytes).hexdigest() != EXPECTED_WORKER_SHA256:
        raise ValueError('Worker không khớp phiên bản notebook; xuất lại bundle/notebook cùng phiên bản')
    manifest = json.loads(archive.read('manifest.json'))
WORKER = Path('/content/offline_e5_colab.py')
WORKER.write_bytes(worker_bytes)
JOB_DIR = Path('/content/drive/MyDrive/HanoiLandLawEmbeddings') / manifest['input_sha256']
OUTPUT = JOB_DIR / 'colab-e5-complete.zip'
print('Số đoạn:', manifest['chunks'], '| Corpus:', manifest['source_release'])
print('Checkpoint:', JOB_DIR)


## 3. Chạy embedding trên CPU

Chạy lại cell này để tiếp tục phần còn thiếu. Không đóng phiên chủ động khi job đang chạy. Model và tokenizer tải khoảng 123 MB trong lần đầu; không tải toàn bộ repository model.


In [ ]:
import os, subprocess, sys
threads = min(4, max(1, os.cpu_count() or 2))
command = [sys.executable, '-u', str(WORKER), 'run', str(BUNDLE),
           '--work-dir', str(JOB_DIR), '--output', str(OUTPUT),
           '--model-dir', '/content/landlaw-e5-model', '--threads', str(threads)]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
if process.wait() != 0:
    raise RuntimeError('Job bị dừng. Checkpoint đã lưu; đọc lỗi rồi chạy lại cell để tiếp tục.')


## 4. Kiểm tra và tải kết quả

Chỉ kết quả đủ vector cho toàn bộ đầu vào mới được tạo thành `colab-e5-complete.zip`. Kết quả và checkpoint vẫn nằm trên Drive nếu thao tác download bị ngắt. Gửi artifact này về workspace để kiểm tra/import, không gửi khóa API.


In [ ]:
subprocess.run([sys.executable, str(WORKER), 'validate', str(BUNDLE), str(OUTPUT)], check=True)
files.download(str(OUTPUT))
